# Fitting Bayesian observer models to Laquitaine data — **non-circular (Gaussian) twin**

Line-for-line companion to `earth_laquitaine_fit.ipynb`, but with the **circle removed**. Here the observer is built from ordinary **Gaussians on a straight direction axis** — **no von Mises, no `kappa`**. The width parameter is a plain standard deviation `sigma` (`kappa ≈ 1/sigma²`, so this is the same model in different clothes).

Why keep it? To make the role of the circular machinery obvious by contrast:
- The **sensory-precision** and **prior-strength** parameters are *equally central* here — they are just named `sigma_llh` / `sigma_prior` instead of `kappa_llh` / `kappa_prior`.
- The prior-dynamics comparison (**static vs online vs warm-up**) is *orthogonal* to circular-vs-Gaussian, so all three models survive the swap unchanged.

**Caveat (why the real study is circular).** A Gaussian on a line ignores wrap (359° ≈ 1°). This is a fine approximation when stimuli sit far from 0/360 — subject 1's 80° block clusters around 225°, so it holds well — but biases near the wrap would come out wrong. That single failure is the reason Laquitaine uses von Mises.

In [ ]:
# @title Dependencies + data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import os, requests

URL = "https://github.com/steevelaquitaine/projInference/raw/gh-pages/data/csv/data01_direction4priors.csv"
CSV = "data01_direction4priors.csv"
if not os.path.exists(CSV):
    r = requests.get(URL)
    assert r.status_code == 200, "download failed; contact steeve.laquitaine@epfl.ch"
    open(CSV, "wb").write(r.content)

data = pd.read_csv(CSV)
print(data.shape)
data[["subject_id", "prior_std", "motion_coherence", "motion_direction"]].head()

## The Gaussian observer (non-circular)

Same grid, same logic as the von Mises version — every von Mises is swapped for a Gaussian and every circular op for a linear one:

| circular version | this version |
|---|---|
| `kernel(kappa)` (von Mises rows) | `gauss_kernel(sigma)` (Gaussian rows) |
| `vm_row(mu, kappa)` | `g_row(mu, sigma)` |
| `circular_mean` | plain `np.average` |
| `signed_diff` (wrapped) | plain `target - current` |

`gauss_kernel(sigma)` is still an `n×n` matrix whose row *i* is a bump centred on grid point *i*; it is still symmetric, so it doubles as likelihood and `P(measurement | stimulus)` — the only change is the bump shape.

In [ ]:
# @title Grid + Gaussian observer building blocks
GRID_STEP = 4                      # deg; smaller = more accurate, slower
xg = np.arange(1, 361, GRID_STEP)  # direction grid (treated as a straight line here)
n = len(xg)

def to_idx(val):
    return np.clip(np.round((np.asarray(val, float) - xg[0]) / GRID_STEP).astype(int), 0, n - 1)

def sig(z):    return 1.0 / (1.0 + np.exp(-z))
def logit(p):  return np.log(p / (1.0 - p))

def gauss_kernel(sigma):
    """n x n Gaussian kernel; row i is a distribution centred on xg[i] (no wrap)."""
    d = xg[:, None] - xg[None, :]
    K = np.exp(-0.5 * (d / sigma) ** 2)
    return K / K.sum(1, keepdims=True)

def g_row(mu_deg, sigma):
    v = np.exp(-0.5 * ((mu_deg - xg) / sigma) ** 2)
    return v / v.sum()

def response_prob(prior, L, stim_idx, lapse):
    """P(estimate | stimulus), marginalizing over the hidden measurement."""
    R = (prior[None, :] * L).argmax(1)              # MAP response idx per measurement
    resp = np.bincount(R, weights=L[stim_idx], minlength=n)
    resp = (1 - lapse) * resp / resp.sum() + lapse / n
    return resp

## Peek inside: the Gaussian kernel and prior

Same three checks as the circular notebook, now with Gaussians. Note the width is `sigma` (bigger = *less* precise) — the exact inverse feel of `kappa`.

In [ ]:
# @title Visualize gauss_kernel + prior width
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

K = gauss_kernel(15.0)
im = ax[0].imshow(K, origin="lower", extent=[xg[0], xg[-1], xg[0], xg[-1]], aspect="auto")
ax[0].set_title("gauss_kernel(sigma=15): each row a Gaussian bump")
ax[0].set_xlabel("evaluated direction (deg)"); ax[0].set_ylabel("center direction (deg)")
fig.colorbar(im, ax=ax[0], shrink=0.8)

for sg in [8, 15, 30, 60]:
    ax[1].plot(xg, g_row(225, sg), label=f"sigma_prior={sg} deg")
ax[1].axvline(225, color="gray", ls=":")
ax[1].set_title("prior = Gaussian at 225 deg; smaller sigma = stronger")
ax[1].set_xlabel("direction (deg)"); ax[1].set_ylabel("prior probability"); ax[1].legend()
plt.tight_layout(); plt.show()

i = to_idx(180)
print("symmetric? max |row_i - col_i| =", np.abs(K[i] - K[:, i]).max())

## Peek inside: `response_prob` and the coherence effect (Gaussian)

Same experimental signature without the circle: lower sensory precision (**larger** `sigma_llh`, i.e. lower coherence) slides `P(estimate | stimulus)` toward the 225° prior.

In [ ]:
# @title P(estimate | stimulus) at three coherences (sigma_llh)
stim_deg = 160.0
s_i = to_idx(stim_deg)
prior = g_row(225, 25.0)

plt.figure(figsize=(9, 4))
for sg, lab, col in [(40.0, "low coh", "tab:blue"),
                     (22.0, "mid coh", "tab:green"),
                     (12.0, "high coh", "tab:red")]:
    L = gauss_kernel(sg)
    resp = response_prob(prior, L, s_i, lapse=0.02)
    mean_est = np.average(xg, weights=resp)
    plt.plot(xg, resp, color=col, label=f"{lab} (sigma_llh={sg}) -> mean {mean_est:.0f} deg")
plt.axvline(stim_deg, color="k", ls="--", label=f"stimulus {stim_deg:.0f} deg")
plt.axvline(225, color="gray", ls=":", label="prior 225 deg")
plt.title("larger sigma_llh (lower coherence) -> estimate slides toward prior")
plt.xlabel("estimate (deg)"); plt.ylabel("P(estimate | stimulus)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## Negative log likelihood for each model

Identical structure to the circular notebook; `kappa → sigma` everywhere and sigmas are fit in log space (they must stay positive).

| Model | prior each trial | free params |
|---|---|---|
| **static** | `g(225, sigma_prior)`, fixed | `sigma_llh`×3 coherence, `sigma_prior`, `lapse` |
| **online** | mean drifts toward each stimulus (linear delta rule, rate `LR`) | static + `LR` |
| **warm-up** | flat until trial `N`, then `g(mean(first N stimuli), sigma_prior)` | static + `N` (grid-searched) |

In [ ]:
# @title NLL functions (Gaussian)
def unpack(theta):
    s = np.exp(theta[:3])          # sigma_llh per coherence level
    sp = np.exp(theta[3])          # sigma_prior
    lap = sig(theta[4])            # lapse
    return s, sp, lap

def nll_static(theta, S, E, C):
    s, sp, lap = unpack(theta)
    prior = g_row(225, sp)
    ll = 0.0
    for c in range(3):
        L = gauss_kernel(s[c])
        R = (prior[None, :] * L).argmax(1)
        m = C == c
        for s_i in np.unique(S[m]):
            resp = np.bincount(R, weights=L[s_i], minlength=n)
            resp = (1 - lap) * resp / resp.sum() + lap / n
            e = E[m & (S == s_i)]
            ll += np.log(resp[e] + 1e-12).sum()
    return -ll

def nll_online(theta, S, E, C, Sdeg, order):
    s, sp, lap = unpack(theta)
    lr = sig(theta[5])
    Ls = [gauss_kernel(s[c]) for c in range(3)]
    mu = 225.0
    ll = 0.0
    for t in order:
        L = Ls[C[t]]
        prior = g_row(mu, sp)
        resp = response_prob(prior, L, S[t], lap)
        ll += np.log(resp[E[t]] + 1e-12)
        mu = mu + lr * (Sdeg[t] - mu)      # linear delta rule (no wrap)
    return -ll

def nll_warmup(theta, S, E, C, Sdeg, order, N):
    s, sp, lap = unpack(theta)
    mu = Sdeg[order[:N]].mean()
    Ls = [gauss_kernel(s[c]) for c in range(3)]
    prior_fix = g_row(mu, sp)
    flat = np.ones(n) / n
    Rfix = [(prior_fix[None, :] * Ls[c]).argmax(1) for c in range(3)]
    Rflat = [(flat[None, :] * Ls[c]).argmax(1) for c in range(3)]
    ll = 0.0
    for i, t in enumerate(order):
        L = Ls[C[t]]
        R = Rflat[C[t]] if i < N else Rfix[C[t]]
        resp = np.bincount(R, weights=L[S[t]], minlength=n)
        resp = (1 - lap) * resp / resp.sum() + lap / n
        ll += np.log(resp[E[t]] + 1e-12)
    return -ll

def simulate_static(theta, S, C, rng):
    """synthetic estimates from the static Gaussian model (parameter recovery)."""
    s, sp, lap = unpack(theta)
    prior = g_row(225, sp)
    Ls = [gauss_kernel(s[c]) for c in range(3)]
    Rs = [(prior[None, :] * Ls[c]).argmax(1) for c in range(3)]
    E = np.zeros(len(S), int)
    for i in range(len(S)):
        c = C[i]
        resp = np.bincount(Rs[c], weights=Ls[c][S[i]], minlength=n)
        resp = (1 - lap) * resp / resp.sum() + lap / n
        E[i] = rng.choice(n, p=resp)
    return E

def fit(nll, x0, args, maxiter=1500):
    return minimize(nll, x0, args=args, method="Nelder-Mead",
                    options={"maxiter": maxiter, "xatol": 1e-2, "fatol": 1e-2})

## Prepare one subject, one prior block

Same slice as the circular notebook: **subject 1**, the **80° prior block**, three coherences, trials in temporal order for the online / warm-up models.

In [ ]:
# @title Build arrays for subject 1, prior_std = 80
SUBJECT_ID = 1
PRIOR_STD = 80

sub = data[(data.subject_id == SUBJECT_ID) & (data.prior_std == PRIOR_STD)].copy()
sub = sub.dropna(subset=["estimate_x", "estimate_y"])
sub = sub.sort_values(["session_id", "run_id", "trial_index"]).reset_index(drop=True)

est_deg = np.round(np.degrees(np.arctan2(sub.estimate_y.values, sub.estimate_x.values)) % 360)
Sdeg = sub.motion_direction.values.astype(float)
coh = sub.motion_coherence.values
coh_levels = np.sort(np.unique(coh))

S = to_idx(Sdeg)
E = to_idx(est_deg)
C = np.searchsorted(coh_levels, coh)
order = np.arange(len(S))

print(f"{len(S)} trials | coherences {coh_levels} | unique stimuli {np.unique(Sdeg)}")

## Sanity check: recover known parameters from synthetic data

Same rule as the circular notebook — confirm the fitter recovers parameters it generated before trusting real-data fits. Note the sigmas *fall* with coherence (opposite direction to kappa).

In [ ]:
# @title Parameter recovery (static Gaussian model)
rng = np.random.default_rng(0)
true_theta = np.array([np.log(40.0), np.log(22.0), np.log(12.0),  # sigma_llh (falls with coherence)
                       np.log(25.0),                              # sigma_prior
                       logit(0.05)])                              # lapse
E_syn = simulate_static(true_theta, S, C, rng)

x0 = np.array([np.log(50.0), np.log(30.0), np.log(18.0), np.log(30.0), logit(0.1)])
res_syn = fit(nll_static, x0, (S, E_syn, C))

def show(theta):
    s, sp, lap = unpack(theta)
    return f"sllh={np.round(s,1)}  sprior={sp:.1f}  lapse={lap:.3f}"

print("true     :", show(true_theta))
print("recovered:", show(res_syn.x))

## Fit the three models to the real data & compare with AIC

`AIC = 2k + 2·NLL`; lower is better. Identical comparison to the circular notebook — the point is to see whether the static-vs-online conclusion survives dropping the circle.

In [ ]:
# @title Fit static / online / warm-up, then compare
x0s = np.array([np.log(50.0), np.log(30.0), np.log(18.0), np.log(30.0), logit(0.1)])

res_static = fit(nll_static, x0s, (S, E, C))
k_static = 5
aic_static = 2 * k_static + 2 * res_static.fun

x0o = np.append(res_static.x, logit(0.05))
res_online = fit(nll_online, x0o, (S, E, C, Sdeg, order))
k_online = 6
aic_online = 2 * k_online + 2 * res_online.fun

best = None
for N in [10, 20, 40, 80]:
    r = fit(nll_warmup, res_static.x, (S, E, C, Sdeg, order, N))
    if best is None or r.fun < best[1].fun:
        best = (N, r)
N_best, res_warm = best
k_warm = 6
aic_warm = 2 * k_warm + 2 * res_warm.fun

rows = [("static", k_static, res_static.fun, aic_static),
        ("online", k_online, res_online.fun, aic_online),
        (f"warm-up (N={N_best})", k_warm, res_warm.fun, aic_warm)]
print(f"{'model':<18}{'k':>3}{'NLL':>12}{'AIC':>12}")
for name, k, nllv, aic in rows:
    print(f"{name:<18}{k:>3}{nllv:>12.1f}{aic:>12.1f}")
winner = min(rows, key=lambda r: r[3])[0]
print(f"\nlowest AIC -> {winner}")

## Read out and visualize the winning fit

Predicted vs observed mean estimate per stimulus and coherence — plain (linear) means this time.

In [ ]:
# @title Predicted vs observed mean estimate (static Gaussian fit)
s, sp, lap = unpack(res_static.x)
prior = g_row(225, sp)

plt.figure(figsize=(12, 4))
for c in range(3):
    L = gauss_kernel(s[c])
    R = (prior[None, :] * L).argmax(1)
    us = np.unique(S[C == c])
    pred_mean, data_mean, stim_deg = [], [], []
    for s_i in us:
        resp = np.bincount(R, weights=L[s_i], minlength=n)
        resp = (1 - lap) * resp / resp.sum() + lap / n
        pred_mean.append(np.average(xg, weights=resp))
        e = E[(C == c) & (S == s_i)]
        data_mean.append(xg[e].mean())
        stim_deg.append(xg[s_i])
    ax = plt.subplot(1, 3, c + 1)
    order_s = np.argsort(stim_deg)
    stim_deg = np.array(stim_deg)[order_s]
    ax.plot(stim_deg, np.array(data_mean)[order_s], "o", color=[0.5, 0, 0], label="data")
    ax.plot(stim_deg, np.array(pred_mean)[order_s], "-", color="k", label="model")
    ax.axhline(225, color="gray", ls=":")
    ax.plot([stim_deg.min(), stim_deg.max()], [stim_deg.min(), stim_deg.max()], "r--", lw=1)
    ax.set_title(f"{int(coh_levels[c]*100)}% coherence")
    ax.set_xlabel("stimulus dir (deg)")
    if c == 0:
        ax.set_ylabel("mean estimate (deg)"); ax.legend()
plt.tight_layout(); plt.show()

## Notes: Gaussian vs von Mises

- **What stayed the same.** Whole pipeline: grid → kernel → Bayes MAP → marginalize → NLL → fit → AIC. The sensory-precision and prior-strength parameters remain the two load-bearing knobs — renamed `sigma_*`, not removed. Confirms the earlier point: *dropping the circle drops the name `kappa`, not its importance.*
- **What changed.** `sigma` has the opposite orientation to `kappa` (`kappa ≈ 1/sigma²`); the delta rule and the readout means are linear.
- **Where it breaks.** Any stimulus or estimate near 0°/360° is mishandled because a line has no wrap. If the AIC ranking here disagrees with the circular notebook, wrap effects are the prime suspect — which is precisely why the real study is circular. See `earth_laquitaine_fit.ipynb` for the von Mises original.